# (g,s)-Dependent LOLR Scenario Tables

This notebook loads the saved solved bundles from the main `v2` scenario and the auxiliary test notebooks, simulates each case, and writes combined LaTeX tables.

In [1]:
using Serialization
using Printf
include("src/simple_s_lolr_v2.jl")

function _v2_notebook_dir()
    return isfile("src/simple_s_lolr_v2.jl") ? pwd() : joinpath(pwd(), "1period_s_lolr_v2")
end

function _v2_transition_moments(sim, model; burnin = 10_000)
    T = length(sim.default)
    @assert 1 <= burnin < T
    high_idx = argmax(model.g)
    low_idx = argmin(model.g)
    starts = falses(T)
    gh_to_gl = falses(T)
    nd_before = falses(T)

    for t in 2:T
        starts[t] = sim.default[t] && !sim.default[t - 1]
        gh_to_gl[t] = sim.g_idx[t - 1] == high_idx && sim.g_idx[t] == low_idx
        nd_before[t] = !sim.default[t - 1]
    end

    sample = (burnin + 1):T
    default_starts = starts[sample]
    gh_to_gl_sample = gh_to_gl[sample]
    nd_before_sample = nd_before[sample]
    coincident = default_starts .& gh_to_gl_sample
    denom_default_starts = sum(default_starts)
    denom_transitions = sum(gh_to_gl_sample .& nd_before_sample)

    pct_default_starts_ghgl = denom_default_starts > 0 ? 100 * sum(coincident) / denom_default_starts : NaN
    pct_ghgl_end_default = denom_transitions > 0 ? 100 * sum(coincident) / denom_transitions : NaN

    return (
        pct_default_starts_ghgl = pct_default_starts_ghgl,
        pct_ghgl_end_default = pct_ghgl_end_default,
    )
end

function _sigma_header_piece(header_bottom)
    m = match(r"\\sigma=([0-9]+)\\%", header_bottom)
    return isnothing(m) ? "" : raw"$\sigma=" * m.captures[1] * raw"\%$"
end

function _delta_only_header(header_bottom)
    return replace(header_bottom, r"^\$\\sigma=[0-9]+\\%,\\ " => raw"$")
end

function _write_v2_moments_table(dst_path, scenario_stats)
    headers_top = [begin
        sigma_piece = _sigma_header_piece(x.scenario.header_bottom)
        isempty(sigma_piece) ? x.scenario.header_top : x.scenario.header_top * ", " * sigma_piece
    end for x in scenario_stats]
    headers_bottom = [_delta_only_header(x.scenario.header_bottom) for x in scenario_stats]

    row_specs = [
        ("avg(spread)", :avg_spread),
        ("avg(qb/y)", :avg_qb_to_y),
        ("avg(f/y)", :avg_f_to_y),
        ("avg(n/y)", :avg_n_to_y),
        ("avg(n_l/y)", :avg_nl_to_y),
        ("avg(l/y)", :avg_l_to_y),
        ("avg(b/y)", :avg_b_to_y),
        ("avg(tb/y)", :avg_tb_to_y),
        ("default rate", :default_rate),
    ]

    blocks = [
        ("First moments (percent)", :all),
        ("Low-growth state (percent)", :low),
        ("High-growth state (percent)", :high),
    ]

    fmt4(x) = isnan(x) ? "NaN" : @sprintf("%.4f", x)
    n = length(scenario_stats)
    colspec = "l||" * join(fill("c", n), "|")
    rowbreak = "\\\\"

    lines = String[]
    push!(lines, raw"\begin{tabular}{" * colspec * "}")
    push!(lines, raw"\toprule")
    push!(lines, "Moment & " * join([raw"\multicolumn{1}{c}{" * h * "}" for h in headers_top], " & ") * " " * rowbreak)
    push!(lines, " & " * join(headers_bottom, " & ") * " " * rowbreak)
    push!(lines, raw"\midrule")

    for (block_label, block_key) in blocks
        push!(lines, raw"\multicolumn{" * string(n + 1) * raw"}{l}{\textit{" * block_label * raw"}}" * " " * rowbreak)
        for (row_label, field) in row_specs
            vals = [fmt4(getproperty(getproperty(x.moments, block_key), field)) for x in scenario_stats]
            push!(lines, row_label * " & " * join(vals, " & ") * " " * rowbreak)
        end
        push!(lines, raw"\midrule")
    end

    lines[end] = raw"\bottomrule"
    push!(lines, raw"\end{tabular}")
    mkpath(dirname(dst_path))
    write(dst_path, join(lines, "\n") * "\n")
    return dst_path
end

function _write_v2_transition_table(dst_path, scenario_stats)
    headers_top = [begin
        sigma_piece = _sigma_header_piece(x.scenario.header_bottom)
        isempty(sigma_piece) ? x.scenario.header_top : x.scenario.header_top * ", " * sigma_piece
    end for x in scenario_stats]
    headers_bottom = [_delta_only_header(x.scenario.header_bottom) for x in scenario_stats]
    fmt2(x) = isnan(x) ? "NaN" : @sprintf("%.2f", x)
    n = length(scenario_stats)
    colspec = "l||" * join(fill("c", n), "|")
    rowbreak = "\\\\"

    rows = [
        raw"\% of default starts by $g_H \to g_L$" => (x -> fmt2(x.transition.pct_default_starts_ghgl)),
        raw"\% of $g_H \to g_L$ ends in default" => (x -> fmt2(x.transition.pct_ghgl_end_default)),
    ]

    lines = String[]
    push!(lines, raw"\begin{tabular}{" * colspec * "}")
    push!(lines, raw"\toprule")
    push!(lines, "Moment & " * join([raw"\multicolumn{1}{c}{" * h * "}" for h in headers_top], " & ") * " " * rowbreak)
    push!(lines, " & " * join(headers_bottom, " & ") * " " * rowbreak)
    push!(lines, raw"\midrule")
    for (label, f) in rows
        vals = [f(x) for x in scenario_stats]
        push!(lines, label * " & " * join(vals, " & ") * " " * rowbreak)
    end
    push!(lines, raw"\bottomrule")
    push!(lines, raw"\end{tabular}")
    mkpath(dirname(dst_path))
    write(dst_path, join(lines, "\n") * "\n")
    return dst_path
end


v2_notebook_dir = _v2_notebook_dir()
v2_result_dir = joinpath(v2_notebook_dir, "result")

scenario_bundle_paths = [
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma8_delta20_else5.jls"),
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma8_delta15_else0.jls"),
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma8_delta10_else0.jls"),
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma4_delta20_else5.jls"),
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma4_delta15_else0.jls"),
    joinpath(v2_result_dir, "s_lolr_v2_case_sigma4_delta30_else0.jls"),
]

scenario_bundles = [open(deserialize, path) for path in scenario_bundle_paths]
scenario_stats = map(scenario_bundles) do bundle
    if hasproperty(bundle, :model) && hasproperty(bundle, :sol)
        sim = simulate(bundle.model, bundle.sol; T = 200_000, seed = 1234)
        moments = summarize_simulation(sim, bundle.model; sol = bundle.sol, burnin = 10_000)
        transition = _v2_transition_moments(sim, bundle.model; burnin = 10_000)
        (scenario = bundle.scenario, moments = moments, transition = transition)
    elseif hasproperty(bundle, :moments) && hasproperty(bundle, :transition)
        (scenario = bundle.scenario, moments = bundle.moments, transition = bundle.transition)
    else
        error("Unsupported scenario bundle format: $(propertynames(bundle))")
    end
end
[(x.scenario.name, x.scenario.bundle_name) for x in scenario_stats]

6-element Vector{Tuple{String, String}}:
 ("Main (σ=8, ΔLB=20, Δelse=5)", "s_lolr_v2_case_sigma8_delta20_else5.jls")
 ("σ=8, ΔLB=15, Δelse=0", "s_lolr_v2_case_sigma8_delta15_else0.jls")
 ("σ=8, ΔLB=10, Δelse=0", "s_lolr_v2_case_sigma8_delta10_else0.jls")
 ("σ=4, ΔLB=20, Δelse=5", "s_lolr_v2_case_sigma4_delta20_else5.jls")
 ("σ=4, ΔLB=15, Δelse=0", "s_lolr_v2_case_sigma4_delta15_else0.jls")
 ("σ=4, ΔLB=30, Δelse=0", "s_lolr_v2_case_sigma4_delta30_else0.jls")

## Combined Moments Table

In [2]:
moments_path = _write_v2_moments_table(
    joinpath(v2_result_dir, "s_lolr_v2_all_moments_table.tex"),
    scenario_stats,
)
println(moments_path)

/Users/erfanahar/Dropbox/Research/Sovereign Debt and Stagnation/1period_s_lolr_v2/result/s_lolr_v2_all_moments_table.tex


## Combined Transition Table

In [3]:
transition_path = _write_v2_transition_table(
    joinpath(v2_result_dir, "s_lolr_v2_all_transition_table.tex"),
    scenario_stats,
)
println(transition_path)

/Users/erfanahar/Dropbox/Research/Sovereign Debt and Stagnation/1period_s_lolr_v2/result/s_lolr_v2_all_transition_table.tex
